# Customer Churn — Exploratory Data Analysis
Run this after placing `telco.csv` in `../data/raw/`.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../data/raw/telco.csv")
df.info()

In [ ]:
# Overall churn rate
df['Churn'].value_counts(normalize=True)

In [ ]:
# Churn split visually
sns.countplot(x='Churn', data=df)
plt.title('Churn distribution')
plt.show()

In [ ]:
# Does contract type affect churn?
sns.countplot(x='Contract', hue='Churn', data=df)
plt.title('Churn by contract type')
plt.xticks(rotation=15)
plt.show()

In [ ]:
# Does tenure affect churn?
sns.histplot(data=df, x='tenure', hue='Churn', kde=True)
plt.title('Tenure distribution by churn status')
plt.show()

In [ ]:
# Correlation of numeric features with churn
df_corr = df.copy()
df_corr['Churn_binary'] = df_corr['Churn'].map({'Yes': 1, 'No': 0})
df_corr.corr(numeric_only=True)['Churn_binary'].sort_values()

# Telco Customer Churn Prediction: Project Report

## 1. The Data
The project uses the **Telco Customer Churn** dataset (sourced from Kaggle). It contains information about a telecommunications company's customers and whether they have churned (left the service).

**Key Information Included in the Data:**
- **Customer Demographics:** Gender, Senior Citizen status, Partner, Dependents.
- **Account Information:** Tenure (months), Contract type (Month-to-month, One year, Two year), Payment Method, Paperless Billing.
- **Services Subscribed:** Phone Service, Multiple Lines, Internet Service (DSL, Fiber optic, No), Online Security, Online Backup, Device Protection, Tech Support, Streaming TV, Streaming Movies.
- **Financial Metrics:** Monthly Charges, Total Charges.
- **Target Variable:** `Churn` (Yes/No) - indicates whether the customer left within the last month.

## 2. What Was Done to the Data

### Exploratory Data Analysis (EDA)
Initial analysis (`notebooks/01_eda.ipynb`) involved looking at the overall churn rate and visualizing how features like contract type and tenure relate to churn. Correlation analysis was performed to identify numeric features associated with churn.

### Data Cleaning (`src/preprocess.py`)
- The `customerID` column was removed as it has no predictive value.
- The `TotalCharges` column, which was originally text, was converted to numeric format. Blank values for brand-new customers (0 months tenure) were filled with `0`.
- The target `Churn` column was mapped from "Yes"/"No" to `1`/`0`.

### Feature Engineering (`src/preprocess.py`)
To give the model stronger signals, several new features were created:
1. **Average Monthly Spend (`avg_monthly_spend`)**: Calculated as `TotalCharges / (tenure + 1)`. Acts as a proxy for the customer's "value density".
2. **Tenure Buckets (`tenure_bucket`)**: Grouped continuous tenure into categorical ranges (`0-1yr`, `1-2yr`, `2-4yr`, `4-6yr`) to capture non-linear relationship with churn.
3. **Engagement Score (`num_services`)**: Counted the total number of services a customer is subscribed to (out of 9 possible services).
4. **One-Hot Encoding**: All categorical variables (including the new `tenure_bucket`) were converted into binary columns (dummy variables) suitable for machine learning algorithms.

### Machine Learning Modeling (`src/train.py`)
1. **Data Split**: Data was split into 80% training and 20% testing, using stratification to ensure the churn ratio remained consistent in both sets.
2. **Handling Class Imbalance**: **SMOTE** (Synthetic Minority Over-sampling Technique) was applied **only** to the training data to artificially balance the churn/non-churn examples, helping the model learn the minority class better.
3. **Model Selection**: Three models were trained and compared: Logistic Regression, Random Forest, and XGBoost.
4. **Hyperparameter Tuning**: XGBoost was selected and fine-tuned using `RandomizedSearchCV` across multiple parameters (trees, depth, learning rate) to maximize the ROC-AUC score.
5. **Interpretability**: Used **SHAP** (SHapley Additive exPlanations) to understand which features most strongly drive the model's predictions.

### Business Impact Simulation (`src/business_impact.py`)
1. **Risk Segmentation**: The test set was scored and segmented into risk tiers: Low (<30% probability), Medium (30%-60%), and High (>60%).
2. **Retention Campaign**: Simulated the business value of acting on these predictions. Assuming a targeted intervention (e.g., a discount) reduces churn probability by 30% among "High" risk customers, it estimates how many churners could be prevented and the overall attrition reduction.

## 3. The Outputs
The pipeline generates the following assets:

- **`models/churn_model.pkl`**: The finalized, trained XGBoost model.
- **`models/feature_columns.pkl`**: A list preserving the exact feature column order, ensuring the Streamlit app passes data to the model correctly.
- **`reports/model_comparison.csv`**: Performance metrics (Accuracy, Precision, Recall, F1, ROC-AUC) comparing Logistic Regression, Random Forest, and XGBoost.
- **`reports/shap_summary.png`**: A global feature importance chart explaining what drives churn.
- **`reports/scored_customers.csv`**: The complete test set containing each customer's actual outcome, predicted churn probability, and assigned risk tier. This is ready to be loaded into BI tools (Power BI, Tableau).

## 4. Code to Run the Project

You can run the entire pipeline sequentially with the following commands.

Ensure you have placed the raw data file at `data/raw/telco.csv` before starting.

```bash
# 1. Install dependencies
pip install -r requirements.txt

# 2. Run the training pipeline (trains model, generates metrics and SHAP plot)
cd src
python train.py

# 3. Run the business simulation (segments customers, exports scored CSV)
python business_impact.py

# 4. Start the interactive Streamlit Web App
cd ..
streamlit run app/streamlit_app.py
```
